# Mol2Vec


In [ ]:

!pip install pandas==0.23.0
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

!pip install git+https://github.com/samoturk/mol2vec;

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
from mol2vec.features import mol2alt_sentence, MolSentence, sentences2vec
from gensim.models import Word2Vec
from rdkit import Chem  # Import the Chem module

mdf = pd.read_csv('NAI2_Train.csv') # NAI2_Test.csv

smiles = mdf['Smiles'].values

# Define y
y_all = mdf['Class'].values
y_df = pd.DataFrame(y_all)


model = Word2Vec.load("/kaggle/input/datasets/paxpoom/mol2vec100d/model_100dim.pkl")
mdf['mol'] = mdf['Smiles'].apply(lambda x: Chem.MolFromSmiles(x))

# Replace 'model.wv.vocab' with 'model.wv.key_to_index'
def sentences2vec_updated(sentences, model, unseen='UNK'):
    keys = set(model.wv.key_to_index.keys())
    vec = []
    for word in sentences:
        if word in keys:
            vec.append(model.wv[word])
        else:
            vec.append(model.wv[unseen])
    return np.array(vec)  # Convert the list to a numpy array

# Function to apply Mol2Vec to the entire DataFrame and save to CSV
def save_mol2vec_features(mdf, model, output_csv='mol2vec_features.csv'):
    features = []
    for idx, row in mdf.iterrows():
        # Convert SMILES to Mol object
        mol_sentence = mol2alt_sentence(row['mol'], radius=1)
        sentence_obj = MolSentence(mol_sentence)
        
        # Get the Mol2Vec feature vector for the sentence
        vector = sentences2vec_updated(sentence_obj, model, unseen='UNK')

        vector = np.mean(vector, axis=0)

        # Flatten the vector into a 1D array
        features.append(vector.flatten())  # Flatten to a 1D array for each molecule
    
    # Convert list of feature vectors into a DataFrame
    feature_df = pd.DataFrame(features)
    
    # Save the DataFrame to a CSV file
    feature_df = pd.concat([feature_df, y_df], axis=1)
    feature_df.to_csv(output_csv, index=True)
    print(f"Mol2Vec features saved to {output_csv}")


save_mol2vec_features(mdf, model, output_csv='R1D100_train.csv')  # Using the 100-dimensional model


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load Data
train_df = pd.read_csv("/kaggle/input/datasets/paxpoom/nai-r1d100/R1D100_train.csv")
test_df = pd.read_csv("/kaggle/input/datasets/paxpoom/nai-r1d100/R1D100_test.csv")

X_train_raw = train_df.iloc[:, 1:-1].values.astype(np.float32)
X_test_raw = test_df.iloc[:, 1:-1].values.astype(np.float32)

y_train = train_df.iloc[:, -1].values
y_test = test_df.iloc[:, -1].values

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

class Mol2VecDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32)

train_dataset = Mol2VecDataset(X_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========================================= BiGRU Autoencoder
# =========================================
class Mol2VecBiGRUAE(nn.Module):
    def __init__(self, seq_len=100, hidden_dim=32, latent_dim=32):
        super().__init__()
        self.seq_len = seq_len

        # Encoder: BiGRU -> [B, hidden_dim * 2]
        self.encoder = nn.GRU(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True 
        )
        self.embedding = nn.Linear(hidden_dim * 2, latent_dim)
        self.norm = nn.LayerNorm(latent_dim)

        # Decoder: BiGRU
        self.decoder = nn.GRU(
            input_size=latent_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True # False
        )
        self.output_layer = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # (B, 100) -> (B, 100, 1)
        x_seq = x.unsqueeze(-1)

        # Encoder
        _, hidden = self.encoder(x_seq) # hidden shape: [2, B, hidden_dim]
        hidden_concat = torch.cat((hidden[0], hidden[1]), dim=1) # [B, hidden_dim * 2]

        z = self.embedding(hidden_concat)
        z = self.norm(z)

        # Decoder
        decoder_input = z.unsqueeze(1).repeat(1, self.seq_len, 1) # [B, 100, latent_dim]
        decoder_output, _ = self.decoder(decoder_input)
        reconstruction = self.output_layer(decoder_output).squeeze(-1) # [B, 100]

        return reconstruction, z

model = Mol2VecBiGRUAE(
    seq_len=X_train.shape[1],
    hidden_dim=32,
    latent_dim=32
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for x in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        reconstruction, embedding = model(x)
        loss = criterion(reconstruction, x)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Loss = {epoch_loss:.6f}")

# Safe Extract Embedding via DataLoader (Prevent GPU OOM)
def safe_extract(model, data_array, batch_size=64):
    model.eval()
    loader = DataLoader(Mol2VecDataset(data_array), batch_size=batch_size, shuffle=False)
    embeddings = []
    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            _, z = model(x)
            embeddings.append(z.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

train_embedding = safe_extract(model, X_train)
test_embedding  = safe_extract(model, X_test)

# Save DF
embedding_columns = [f"GRU_{i+1}" for i in range(train_embedding.shape[1])]
train_embedding_df = pd.DataFrame(train_embedding, columns=embedding_columns)
test_embedding_df  = pd.DataFrame(test_embedding, columns=embedding_columns)

train_embedding_df.insert(0, "ID", train_df.iloc[:, 0].values)
train_embedding_df["Label"] = y_train

test_embedding_df.insert(0, "ID", test_df.iloc[:, 0].values)
test_embedding_df["Label"] = y_test

train_embedding_df.to_csv("Mol2Vec_GRU_train.csv", index=False)
test_embedding_df.to_csv("Mol2Vec_GRU_test.csv", index=False)

print("Train embedding shape :", train_embedding.shape)
print("Test embedding shape  :", test_embedding.shape)